**Setting up PySpark and creating a Spark Session (since this a new Notebook)**

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark

In [ ]:
# Setting the environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [ ]:
# Initialize Spark Session
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = spark.read.option("header", "true").csv("/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/09_Dataset/books.csv")

In [ ]:
df.show()

In [ ]:
#Display the data types of the columns
df.dtypes

# Using SQL to interface with Spark

**Step 1**: Register the DataFrame as a SQL temporary view so that you can interact with it using SQL commands


In [ ]:
df.createOrReplaceTempView("books")

**Step 2**: Query the temporary view using SQL

In [ ]:
sqlDF = spark.sql("SELECT * FROM books")
sqlDF.show()

**Step 3**: Filter the data so that only records where the average rating is greater than 4.5

In [ ]:
sqlDF.filter("average_rating > 4.5").show()

**Step 4**: Filter the dataset to include books that have an average rating of greater than 4.5 and the publisher is 'Ballantine Books'

In [ ]:
sqlDF.filter("average_rating > 4.5").filter("publisher == 'Ballantine Books'").show()

**Step 5**: Add a new column that depicts for each book what proportion of the overall ratings count are text review counts

In [ ]:
import pyspark.sql.functions as F

sqlDF = sqlDF.withColumn("proportion_of_ratings", F.bround((sqlDF.text_reviews_count/sqlDF.ratings_count),2))
sqlDF = sqlDF.withColumn("avg_rating_decimal", sqlDF.average_rating.cast('Decimal(4,2)'))
sqlDF.show()

# Aggregating data using GroupBy

**Step 6**: Find the lowest average_rating where the publisher if 'Broadway Books'

In [ ]:
sqlDF.filter("publisher == 'Broadway Books'").groupBy().min("avg_rating_decimal").show()


**Step 7**: Find the highest average_rating where the publisher if 'Broadway Books'


In [ ]:
sqlDF.filter("publisher == 'Broadway Books'").groupBy().max("avg_rating_decimal").show()

**Step 8**: Ascertain the cumulative count of ratings received for Broadway Books

In [ ]:
#First convert from string type to integer
sqlDF = sqlDF.withColumn("total_rating_count", sqlDF.ratings_count.cast('Integer'))
sqlDF.filter("publisher == 'Broadway Books'").groupBy().sum("total_rating_count").show()

**Step 9:** Ascertain the average  count of ratings received for Broadway Books

In [ ]:
sqlDF.filter("publisher == 'Broadway Books'").groupBy().avg("total_rating_count").show()

**Step 10**: Show average count of ratings for each publisher

In [ ]:
sqlDF.groupBy("publisher").avg("total_rating_count").show()


We will utilize **.agg()** method - this method lets you pass an aggregate column expression that uses any of the aggregate functions from the pyspark.sql.functions submodule.

The *pyspark.sql.functions* submodule includes a number of useful functions for performing a variety of computations like standard deviations. Note that all the aggregation functions in this submodule take the name of a column in a GroupedData table as demonstrated via the example below

**Step 11**: Compute the standard deviation of ratings count by publisher

In [ ]:
# Import pyspark.sql.functions as F
import pyspark.sql.functions as F


# Standard deviation of ratings count by publisher
sqlDF.groupBy("publisher").agg(F.stddev("ratings_count")).show()

# Joining datasets using Spark SQL

In [ ]:
# Create the tables that will be joined

valuesA = [('Dog',1),('Monkey',2),('Elephant',3),('Penguin',4)]
TableA = spark.createDataFrame(valuesA,['name','id'])

valuesB = [('Lizard',1),('Dog',2),('Elephant',3),('Rat',4)]
TableB = spark.createDataFrame(valuesB,['name','id'])

TableA.show()
TableB.show()

In [ ]:
# Create table aliases

T1 = TableA.alias('T1')
T2 = TableB.alias('T2')

In [ ]:
# Inner Join on name - returns matching records from the 2 tables respectively

T1.join(T2, T1.name == T2.name, 'inner').show()


In [ ]:
# Left Join Example - If you want to select all records from table 1 and return data from table 2 when it matches, you choose ‘left’

T1.join(T2, T1.name == T2.name, 'left').show()

In [ ]:
# Left Join - filtering out nulls

left_join = T1.join(T2, T1.name == T2.name, 'left')
left_join.filter((T2.name).isNotNull()).show()

In [ ]:
# Right Join Example - If you want to select all records from table 2 and return data from table 1 when it matches, you choose ‘right’

T1.join(T2, T1.name == T2.name, 'right').show()

In [ ]:
# Full Outer Join - This shows all records from the left table and all the records from the right table and nulls where the two do not match

T1.join(T2, T1.name == T2.name,how='full').show()